# Plasma Waveguide Validation

This notebook validates the plasma waveguide physics model, specifically checking for subluminal group velocities and channel formation conditions.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from plasma_physics import plasma_frequencies, whistler_group_velocity
from waveguide_simulation import WaveguideSimulation
from visualization import plot_group_velocity, plot_channel_formation

%matplotlib inline

In [ ]:
# Parameters
n_e = 1e18  # m^-3
B = 0.5     # T
T_e = 116000 # K (10 eV)

frequencies = np.linspace(0.5e9, 2e9, 100)
E0_range = np.linspace(0, 1e7, 100)

sim = WaveguideSimulation(n_e, B, T_e)

# Run Simulation
vg = sim.simulate_group_velocity(frequencies)
p_pond, p_therm, formed = sim.check_channel_formation(E0_range)

# Validation Checks
c = 3e8
max_vg = np.max(vg)
print(f"Max Group Velocity: {max_vg/c:.4f} c")

if max_vg < c:
    print("✅ Velocity is strictly subluminal.")
else:
    print("❌ WARNING: Superluminal velocity detected!")

threshold_idx = np.where(formed)[0]
if len(threshold_idx) > 0:
    print(f"✅ Channel forms at E > {E0_range[threshold_idx[0]]/1e6:.2f} MV/m")
else:
    print("❌ Channel does not form in this range.")

In [ ]:
# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot(frequencies / 1e9, vg / c)
ax1.set_xlabel('Frequency (GHz)')
ax1.set_ylabel('Group Velocity (c)')
ax1.set_title('Whistler-Mode Group Velocity')
ax1.grid(True)
ax1.axhline(1.0, color='r', linestyle='--')

ax2.plot(E0_range / 1e6, p_pond, label='Ponderomotive Pressure')
ax2.axhline(y=p_therm, color='r', linestyle='--', label='Thermal Pressure')
ax2.set_xlabel('Electric Field (MV/m)')
ax2.set_ylabel('Pressure (Pa)')
ax2.set_title('Channel Formation Condition')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()